Connected to fvc_structure (Python 3.10.18)

In [ ]:
## Works for raster
import json
from pathlib import Path

import pdal
import rasterio
import fiona
from shapely.geometry import shape
from shapely.ops import unary_union, transform as shp_transform
from shapely.validation import make_valid
from shapely.wkt import dumps as wkt_dumps
from pyproj import CRS, Transformer

# === Inputs ===
# input_las = Path("/media/laura/Extreme SSD/qgis/calperumResearch/site1_1_DD0001/lidar/cloud1457eb47b07af181.las")
input_las = Path("/media/laura/laura_usb/uas_data/DD0001/lidar/cloud2d765c57b8471e2c.las")
aoi_raster = Path("/media/laura/Extreme SSD/qgis/calperumResearch/site1_1_DD0001/inputs/predictors/tiles_3072/stacked/tiles_multispectral.26.tif")
aoi_vector = Path("/media/laura/Extreme SSD/qgis/calperumResearch/site1_1_DD0001/inputs/masks/tiles_3072/raw/fvc_class/annotation_shp/mask_fvc_3072.26.shp")

# === Outputs ===
out_dir = Path("/media/laura/laura_usb/uas_data/DD0001/lidar/trim")
out_dir.mkdir(parents=True, exist_ok=True)
raster_clip_out = out_dir / (input_las.stem + "_clip_raster_extent.las")
shp_clip_out    = out_dir / (input_las.stem + "_clip_shp_polygon.las")

# === Get raster CRS + bounds ===
with rasterio.open(aoi_raster) as src:
    raster_crs = CRS.from_wkt(src.crs.to_wkt())
    target_epsg = raster_crs.to_epsg()
    minx, miny, maxx, maxy = src.bounds

# Slightly pad bounds to avoid edge numeric issues
pad = max((maxx - minx), (maxy - miny)) * 1e-9
bounds_str = f"([{minx - pad},{maxx + pad}],[{miny - pad},{maxy + pad}])"

print(f"Raster EPSG: {target_epsg}")
print(f"Raster bounds: {minx:.3f}, {miny:.3f}, {maxx:.3f}, {maxy:.3f}")

# === Read shapefile geometry, unify, reproject → raster CRS, clean ===
with fiona.open(aoi_vector, "r") as src:
    if src.crs_wkt:
        shp_crs = CRS.from_wkt(src.crs_wkt)
    elif src.crs:
        shp_crs = CRS.from_user_input(src.crs)
    else:
        shp_crs = raster_crs
    geoms = [shape(feat["geometry"]) for feat in src]

poly = unary_union(geoms)

# Fix geometry, drop Z/M
poly = make_valid(poly).buffer(0)
if not shp_crs == raster_crs:
    transformer = Transformer.from_crs(shp_crs, raster_crs, always_xy=True)
    poly = shp_transform(lambda x, y, z=None: transformer.transform(x, y), poly)
poly = shp_transform(lambda x, y, z=None: (x, y), poly)

# Add a small positive buffer to catch edge points (5 cm)
poly = poly.buffer(0.05)

if poly.is_empty:
    raise ValueError("AOI polygon is empty after reprojection/validation.")

minx_p, miny_p, maxx_p, maxy_p = poly.bounds
print(f"AOI polygon area (raster CRS units^2): {poly.area:.3f}")
print(f"AOI bbox: {minx_p:.3f}, {miny_p:.3f}, {maxx_p:.3f}, {maxy_p:.3f}")

# Export WKT (PDAL accepts 'polygon' as WKT)
polygon_wkt = wkt_dumps(poly, rounding_precision=6)

# === Build PDAL pipeline (branch to 2 outputs) ===
stages = []

# 1) Read LAS
stages.append({"type": "readers.las", "filename": str(input_las), "tag": "src"})

# 2) Reproject LAS → raster CRS
stages.append({
    "type": "filters.reprojection",
    "out_srs": f"EPSG:{target_epsg}",
    "inputs": ["src"],
    "tag": "rpj"
})

# 3) Crop to raster bounds (fast pre-clip)
stages.append({
    "type": "filters.crop",
    "bounds": bounds_str,
    "inputs": ["rpj"],
    "tag": "crop_bounds"
})

# 4) Write: clipped to raster extent
stages.append({
    "type": "writers.las",
    "filename": str(raster_clip_out),
    "minor_version": 4,
    "inputs": ["crop_bounds"]
})

# 5) Crop that branch by polygon (precise)
stages.append({
    "type": "filters.crop",
    "polygon": polygon_wkt,   # WKT in same CRS as LAS (raster CRS)
    "inputs": ["crop_bounds"],
    "tag": "crop_poly"
})

# 6) Write: clipped to shapefile polygon
stages.append({
    "type": "writers.las",
    "filename": str(shp_clip_out),
    "minor_version": 4,
    "inputs": ["crop_poly"]
})

pipeline = {"pipeline": stages}
pipe = pdal.Pipeline(json.dumps(pipeline))
cnt = pipe.execute()

print(f"✅ PDAL points processed: {cnt}")
print(f"✅ Raster-extent clip: {raster_clip_out}")
print(f"✅ Shapefile-polygon clip: {shp_clip_out}")

Raster EPSG: 7854
Raster bounds: 461986.605, 6237690.408, 462017.325, 6237721.128
AOI polygon area (raster CRS units^2): 660.415
AOI bbox: 461986.555, 6237690.357, 462017.375, 6237721.178
✅ PDAL points processed: 28923
✅ Raster-extent clip: /media/laura/laura_usb/uas_data/DD0001/lidar/trim/cloud2d765c57b8471e2c_clip_raster_extent.las
✅ Shapefile-polygon clip: /media/laura/laura_usb/uas_data/DD0001/lidar/trim/cloud2d765c57b8471e2c_clip_shp_polygon.las


In [ ]:
### keep normal
### keep normal (robust + verified)
import json
from pathlib import Path
import subprocess

import pdal
import rasterio
import fiona
from shapely.geometry import shape
from shapely.ops import unary_union, transform as shp_transform
from shapely.validation import make_valid
from shapely.wkt import dumps as wkt_dumps
from pyproj import CRS, Transformer

# === Inputs ===
# input_las = Path("/media/laura/Extreme SSD/qgis/calperumResearch/site1_1_DD0001/lidar/cloud1457eb47b07af181.las")
input_las = Path("/media/laura/laura_usb/uas_data/DD0001/lidar/cloud2d765c57b8471e2c.las")
aoi_raster = Path("/media/laura/Extreme SSD/qgis/calperumResearch/site1_1_DD0001/inputs/predictors/tiles_3072/stacked/tiles_multispectral.26.tif")
aoi_vector = Path("/media/laura/Extreme SSD/qgis/calperumResearch/site1_1_DD0001/inputs/masks/tiles_3072/raw/fvc_class/annotation_shp/mask_fvc_3072.26.shp")

# === Outputs ===
# out_dir = Path("/media/laura/Laura 102/uas_data/site1_1_DD0001/lidar/trim/normals")
out_dir = Path("/media/laura/laura_usb/uas_data/DD0001/lidar/trim")
out_dir.mkdir(parents=True, exist_ok=True)
raster_clip_out = out_dir / (input_las.stem + "_clip_raster_extent.las")
shp_clip_out    = out_dir / (input_las.stem + "_clip_shp_polygon.las")

# === Get raster CRS + bounds ===
with rasterio.open(aoi_raster) as src:
    raster_crs = CRS.from_wkt(src.crs.to_wkt())
    target_epsg = raster_crs.to_epsg()
    minx, miny, maxx, maxy = src.bounds

pad = max((maxx - minx), (maxy - miny)) * 1e-9
bounds_str = f"([{minx - pad},{maxx + pad}],[{miny - pad},{maxy + pad}])"

print(f"Raster EPSG: {target_epsg}")
print(f"Raster bounds: {minx:.3f}, {miny:.3f}, {maxx:.3f}, {maxy:.3f}")

# === Read shapefile geometry, unify, reproject → raster CRS, clean ===
with fiona.open(aoi_vector, "r") as src:
    if src.crs_wkt:
        shp_crs = CRS.from_wkt(src.crs_wkt)
    elif src.crs:
        shp_crs = CRS.from_user_input(src.crs)
    else:
        shp_crs = raster_crs
    geoms = [shape(feat["geometry"]) for feat in src]

poly = unary_union(geoms)
poly = make_valid(poly).buffer(0)  # fix geometry
if shp_crs != raster_crs:
    transformer = Transformer.from_crs(shp_crs, raster_crs, always_xy=True)
    poly = shp_transform(lambda x, y, z=None: transformer.transform(x, y), poly)
poly = shp_transform(lambda x, y, z=None: (x, y), poly)
poly = poly.buffer(0.05)  # small inclusion buffer

if poly.is_empty:
    raise ValueError("AOI polygon is empty after reprojection/validation.")

minx_p, miny_p, maxx_p, maxy_p = poly.bounds
print(f"AOI area: {poly.area:.3f}  bbox: {minx_p:.3f},{miny_p:.3f},{maxx_p:.3f},{maxy_p:.3f}")

polygon_wkt = wkt_dumps(poly, rounding_precision=6)

# === Build PDAL pipeline (bounds clip + normals) -> (polygon clip + normals) ===
stages = []

# 1) Read LAS
stages.append({"type": "readers.las", "filename": str(input_las), "tag": "src"})

# 2) Reproject LAS → raster CRS
stages.append({
    "type": "filters.reprojection",
    "out_srs": f"EPSG:{target_epsg}",
    "inputs": ["src"],
    "tag": "rpj"
})

# 3) Crop to raster bounds
stages.append({
    "type": "filters.crop",
    "bounds": bounds_str,
    "inputs": ["rpj"],
    "tag": "crop_bounds"
})

# 4) Normals for raster-extent branch
stages.append({
    "type": "filters.normal",
    "knn": 16,
    "inputs": ["crop_bounds"],
    "tag": "norm_bounds"
})

# 5) Write: raster extent (preserve structure + extras)
stages.append({
    "type": "writers.las",
    "filename": str(raster_clip_out),
    "forward": "all",
    "extra_dims": "all",
    "inputs": ["norm_bounds"]
})

# 6) Polygon crop from bounds branch
stages.append({
    "type": "filters.crop",
    "polygon": polygon_wkt,
    "inputs": ["crop_bounds"],
    "tag": "crop_poly"
})

# 7) Normals for polygon branch
stages.append({
    "type": "filters.normal",
    "knn": 16,
    "inputs": ["crop_poly"],
    "tag": "norm_poly"
})

# 8) Write: polygon clip (preserve structure + extras)
stages.append({
    "type": "writers.las",
    "filename": str(shp_clip_out),
    "forward": "all",
    "extra_dims": "all",
    "inputs": ["norm_poly"]
})

pipeline = {"pipeline": stages}

# === Execute with verbose logging so we can see PDAL errors if any ===
try:
    pipe = pdal.Pipeline(json.dumps(pipeline))
    cnt = pipe.execute()
    print(f"✅ PDAL points processed: {cnt}")
except RuntimeError as e:
    # Some PDAL builds put useful detail only in CLI verbose; dump JSON & run CLI
    pj = out_dir / "debug_normals_pipeline.json"
    pj.write_text(json.dumps(pipeline, indent=2))
    print(f"⚠️ Pipeline failed in bindings. Saved JSON: {pj}")
    print("Re-running via CLI for detailed error...")
    res = subprocess.run(
        ["pdal", "pipeline", str(pj), "--verbose=6"],
        text=True, capture_output=True
    )
    print("----- PDAL STDOUT -----\n", res.stdout)
    print("----- PDAL STDERR -----\n", res.stderr)
    res.check_returncode()  # will raise with full stderr if non-zero

print(f"✅ Raster-extent clip (with normals): {raster_clip_out}")
print(f"✅ Shapefile-polygon clip (with normals): {shp_clip_out}")

# === Verify normals actually exist in outputs ===
try:
    import laspy
    for p in (raster_clip_out, shp_clip_out):
        las = laspy.read(str(p))
        dims = set(las.point_format.dimension_names)
        has_normals = {"NormalX","NormalY","NormalZ"}.issubset(dims)
        print(f"{p.name}: normals present? {has_normals}  dims={sorted(dims)[:12]} ...")
except Exception as _:
    print("laspy not available for verification; skipping dim check.")

Raster EPSG: 7854
Raster bounds: 461986.605, 6237690.408, 462017.325, 6237721.128
AOI area: 660.415  bbox: 461986.555,6237690.357,462017.375,6237721.178
✅ PDAL points processed: 28923
✅ Raster-extent clip (with normals): /media/laura/laura_usb/uas_data/DD0001/lidar/trim/cloud2d765c57b8471e2c_clip_raster_extent.las
✅ Shapefile-polygon clip (with normals): /media/laura/laura_usb/uas_data/DD0001/lidar/trim/cloud2d765c57b8471e2c_clip_shp_polygon.las
cloud2d765c57b8471e2c_clip_raster_extent.las: normals present? True  dims=['Curvature', 'NormalX', 'NormalY', 'NormalZ', 'X', 'Y', 'Z', 'blue', 'classification', 'edge_of_flight_line', 'gps_time', 'green'] ...
laspy not available for verification; skipping dim check.


Connected to fvc_structure (Python 3.10.18)

In [ ]:
## Works for raster
import json
from pathlib import Path

import pdal
import rasterio
import fiona
from shapely.geometry import shape
from shapely.ops import unary_union, transform as shp_transform
from shapely.validation import make_valid
from shapely.wkt import dumps as wkt_dumps
from pyproj import CRS, Transformer

# === Inputs ===
# input_las = Path("/media/laura/Extreme SSD/qgis/calperumResearch/site1_1_DD0001/lidar/cloud1457eb47b07af181.las")
# input_las = Path("/media/laura/laura_usb/uas_data/DD0001/lidar/cloud2d765c57b8471e2c.las")
input_las = Path("/media/laura/laura_usb/uas_data/DD0001/lidar/cloud2d765c57b8471e2c_7854.las")

aoi_raster = Path("/media/laura/Extreme SSD/qgis/calperumResearch/site1_1_DD0001/inputs/predictors/tiles_3072/stacked/tiles_multispectral.26.tif")
aoi_vector = Path("/media/laura/Extreme SSD/qgis/calperumResearch/site1_1_DD0001/inputs/masks/tiles_3072/raw/fvc_class/annotation_shp/mask_fvc_3072.26.shp")

# === Outputs ===
out_dir = Path("/media/laura/laura_usb/uas_data/DD0001/lidar/trim")
out_dir.mkdir(parents=True, exist_ok=True)
raster_clip_out = out_dir / (input_las.stem + "_clip_raster_extent_7854.las")
shp_clip_out    = out_dir / (input_las.stem + "_clip_shp_polygon.las")

# === Get raster CRS + bounds ===
with rasterio.open(aoi_raster) as src:
    raster_crs = CRS.from_wkt(src.crs.to_wkt())
    target_epsg = raster_crs.to_epsg()
    minx, miny, maxx, maxy = src.bounds

# Slightly pad bounds to avoid edge numeric issues
pad = max((maxx - minx), (maxy - miny)) * 1e-9
bounds_str = f"([{minx - pad},{maxx + pad}],[{miny - pad},{maxy + pad}])"

print(f"Raster EPSG: {target_epsg}")
print(f"Raster bounds: {minx:.3f}, {miny:.3f}, {maxx:.3f}, {maxy:.3f}")

# === Read shapefile geometry, unify, reproject → raster CRS, clean ===
with fiona.open(aoi_vector, "r") as src:
    if src.crs_wkt:
        shp_crs = CRS.from_wkt(src.crs_wkt)
    elif src.crs:
        shp_crs = CRS.from_user_input(src.crs)
    else:
        shp_crs = raster_crs
    geoms = [shape(feat["geometry"]) for feat in src]

poly = unary_union(geoms)

# Fix geometry, drop Z/M
poly = make_valid(poly).buffer(0)
if not shp_crs == raster_crs:
    transformer = Transformer.from_crs(shp_crs, raster_crs, always_xy=True)
    poly = shp_transform(lambda x, y, z=None: transformer.transform(x, y), poly)
poly = shp_transform(lambda x, y, z=None: (x, y), poly)

# Add a small positive buffer to catch edge points (5 cm)
poly = poly.buffer(0.05)

if poly.is_empty:
    raise ValueError("AOI polygon is empty after reprojection/validation.")

minx_p, miny_p, maxx_p, maxy_p = poly.bounds
print(f"AOI polygon area (raster CRS units^2): {poly.area:.3f}")
print(f"AOI bbox: {minx_p:.3f}, {miny_p:.3f}, {maxx_p:.3f}, {maxy_p:.3f}")

# Export WKT (PDAL accepts 'polygon' as WKT)
polygon_wkt = wkt_dumps(poly, rounding_precision=6)

# === Build PDAL pipeline (branch to 2 outputs) ===
stages = []

# 1) Read LAS
stages.append({"type": "readers.las", "filename": str(input_las), "tag": "src"})

# 2) Reproject LAS → raster CRS
stages.append({
    "type": "filters.reprojection",
    "out_srs": f"EPSG:{target_epsg}",
    "inputs": ["src"],
    "tag": "rpj"
})

# 3) Crop to raster bounds (fast pre-clip)
stages.append({
    "type": "filters.crop",
    "bounds": bounds_str,
    "inputs": ["rpj"],
    "tag": "crop_bounds"
})

# 4) Write: clipped to raster extent
stages.append({
    "type": "writers.las",
    "filename": str(raster_clip_out),
    "minor_version": 4,
    "inputs": ["crop_bounds"]
})

# 5) Crop that branch by polygon (precise)
stages.append({
    "type": "filters.crop",
    "polygon": polygon_wkt,   # WKT in same CRS as LAS (raster CRS)
    "inputs": ["crop_bounds"],
    "tag": "crop_poly"
})

# 6) Write: clipped to shapefile polygon
stages.append({
    "type": "writers.las",
    "filename": str(shp_clip_out),
    "minor_version": 4,
    "inputs": ["crop_poly"]
})

pipeline = {"pipeline": stages}
pipe = pdal.Pipeline(json.dumps(pipeline))
cnt = pipe.execute()

print(f"✅ PDAL points processed: {cnt}")
print(f"✅ Raster-extent clip: {raster_clip_out}")
print(f"✅ Shapefile-polygon clip: {shp_clip_out}")

Raster EPSG: 7854
Raster bounds: 461986.605, 6237690.408, 462017.325, 6237721.128
AOI polygon area (raster CRS units^2): 660.415
AOI bbox: 461986.555, 6237690.357, 462017.375, 6237721.178
✅ PDAL points processed: 28923
✅ Raster-extent clip: /media/laura/laura_usb/uas_data/DD0001/lidar/trim/cloud2d765c57b8471e2c_7854_clip_raster_extent_7854.las
✅ Shapefile-polygon clip: /media/laura/laura_usb/uas_data/DD0001/lidar/trim/cloud2d765c57b8471e2c_7854_clip_shp_polygon.las


In [ ]:
## Works for raster
import json
from pathlib import Path

import pdal
import rasterio
import fiona
from shapely.geometry import shape
from shapely.ops import unary_union, transform as shp_transform
from shapely.validation import make_valid
from shapely.wkt import dumps as wkt_dumps
from pyproj import CRS, Transformer

# === Inputs ===
# input_las = Path("/media/laura/Extreme SSD/qgis/calperumResearch/site1_1_DD0001/lidar/cloud1457eb47b07af181.las")
# input_las = Path("/media/laura/laura_usb/uas_data/DD0001/lidar/cloud2d765c57b8471e2c.las")
input_las = Path("/media/laura/laura_usb/uas_data/DD0001/lidar/cloud2d765c57b8471e2c_2d_7854.las")

aoi_raster = Path("/media/laura/Extreme SSD/qgis/calperumResearch/site1_1_DD0001/inputs/predictors/tiles_3072/stacked/tiles_multispectral.26.tif")
aoi_vector = Path("/media/laura/Extreme SSD/qgis/calperumResearch/site1_1_DD0001/inputs/masks/tiles_3072/raw/fvc_class/annotation_shp/mask_fvc_3072.26.shp")

# === Outputs ===
out_dir = Path("/media/laura/laura_usb/uas_data/DD0001/lidar/trim")
out_dir.mkdir(parents=True, exist_ok=True)
raster_clip_out = out_dir / (input_las.stem + "_clip_raster_extent_7854.las")
shp_clip_out    = out_dir / (input_las.stem + "_clip_shp_polygon.las")

# === Get raster CRS + bounds ===
with rasterio.open(aoi_raster) as src:
    raster_crs = CRS.from_wkt(src.crs.to_wkt())
    target_epsg = raster_crs.to_epsg()
    minx, miny, maxx, maxy = src.bounds

# Slightly pad bounds to avoid edge numeric issues
pad = max((maxx - minx), (maxy - miny)) * 1e-9
bounds_str = f"([{minx - pad},{maxx + pad}],[{miny - pad},{maxy + pad}])"

print(f"Raster EPSG: {target_epsg}")
print(f"Raster bounds: {minx:.3f}, {miny:.3f}, {maxx:.3f}, {maxy:.3f}")

# === Read shapefile geometry, unify, reproject → raster CRS, clean ===
with fiona.open(aoi_vector, "r") as src:
    if src.crs_wkt:
        shp_crs = CRS.from_wkt(src.crs_wkt)
    elif src.crs:
        shp_crs = CRS.from_user_input(src.crs)
    else:
        shp_crs = raster_crs
    geoms = [shape(feat["geometry"]) for feat in src]

poly = unary_union(geoms)

# Fix geometry, drop Z/M
poly = make_valid(poly).buffer(0)
if not shp_crs == raster_crs:
    transformer = Transformer.from_crs(shp_crs, raster_crs, always_xy=True)
    poly = shp_transform(lambda x, y, z=None: transformer.transform(x, y), poly)
poly = shp_transform(lambda x, y, z=None: (x, y), poly)

# Add a small positive buffer to catch edge points (5 cm)
poly = poly.buffer(0.05)

if poly.is_empty:
    raise ValueError("AOI polygon is empty after reprojection/validation.")

minx_p, miny_p, maxx_p, maxy_p = poly.bounds
print(f"AOI polygon area (raster CRS units^2): {poly.area:.3f}")
print(f"AOI bbox: {minx_p:.3f}, {miny_p:.3f}, {maxx_p:.3f}, {maxy_p:.3f}")

# Export WKT (PDAL accepts 'polygon' as WKT)
polygon_wkt = wkt_dumps(poly, rounding_precision=6)

# === Build PDAL pipeline (branch to 2 outputs) ===
stages = []

# 1) Read LAS
stages.append({"type": "readers.las", "filename": str(input_las), "tag": "src"})

# 2) Reproject LAS → raster CRS
stages.append({
    "type": "filters.reprojection",
    "out_srs": f"EPSG:{target_epsg}",
    "inputs": ["src"],
    "tag": "rpj"
})

# 3) Crop to raster bounds (fast pre-clip)
stages.append({
    "type": "filters.crop",
    "bounds": bounds_str,
    "inputs": ["rpj"],
    "tag": "crop_bounds"
})

# 4) Write: clipped to raster extent
stages.append({
    "type": "writers.las",
    "filename": str(raster_clip_out),
    "minor_version": 4,
    "inputs": ["crop_bounds"]
})

# 5) Crop that branch by polygon (precise)
stages.append({
    "type": "filters.crop",
    "polygon": polygon_wkt,   # WKT in same CRS as LAS (raster CRS)
    "inputs": ["crop_bounds"],
    "tag": "crop_poly"
})

# 6) Write: clipped to shapefile polygon
stages.append({
    "type": "writers.las",
    "filename": str(shp_clip_out),
    "minor_version": 4,
    "inputs": ["crop_poly"]
})

pipeline = {"pipeline": stages}
pipe = pdal.Pipeline(json.dumps(pipeline))
cnt = pipe.execute()

print(f"✅ PDAL points processed: {cnt}")
print(f"✅ Raster-extent clip: {raster_clip_out}")
print(f"✅ Shapefile-polygon clip: {shp_clip_out}")

Raster EPSG: 7854
Raster bounds: 461986.605, 6237690.408, 462017.325, 6237721.128
